## 04 - Isolation Forest side path

Train an unsupervised Isolation Forest on the same numeric features as CatBoost to produce anomaly scores. Logic: CatBoost says safe but IF anomaly score is very high → trigger human review (zero-day / unknown attacks).

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from catboost import CatBoostClassifier
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import joblib

ROOT = Path.cwd()
MODEL_DIR = ROOT / "fraud_detection" / "models"

with open(MODEL_DIR / "feature_config.json") as f:
    config = json.load(f)
feature_cols = config["feature_cols"]
cat_features = config["cat_features"]

test_df = pd.read_csv(MODEL_DIR / "oot_test_index.csv")
train_df = pd.read_csv(MODEL_DIR / "oot_train_index.csv")
X_train = train_df[feature_cols]
X_test = test_df[feature_cols]
y_test = test_df["is_fraud"]

## Numeric feature subset (IF uses only numerics; categoricals are encoded or dropped)

In [2]:
# Zip is categorical; drop it so IF uses only continuous numeric features
numeric_cols = [c for c in feature_cols if c not in cat_features and c != "customer_zip"]
X_train_num = X_train[numeric_cols].copy()
X_test_num = X_test[numeric_cols].copy()
for c in numeric_cols:
    X_train_num[c] = pd.to_numeric(X_train_num[c], errors="coerce")
    X_test_num[c] = pd.to_numeric(X_test_num[c], errors="coerce")
X_train_num = X_train_num.fillna(-999)
X_test_num = X_test_num.fillna(-999)
print("Numeric features for IF:", numeric_cols)

Numeric features for IF: ['transaction_amount', 'customer_latitude', 'customer_longitude', 'customer_city_population', 'merchant_latitude', 'merchant_longitude', 'is_online', 'distance_to_merchant', 'effective_distance', 'hour_of_day', 'day_of_week', 'tx_count_1h', 'tx_count_24h', 'amt_sum_24h', 'customer_age']


## Train Isolation Forest and get anomaly scores

In [3]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_num)
X_test_scaled = scaler.transform(X_test_num)

iforest = IsolationForest(contamination=0.01, random_state=42, n_estimators=100)
iforest.fit(X_train_scaled)

# decision_function: more negative = more anomalous, more positive = more normal
anom_train = iforest.decision_function(X_train_scaled)
anom_test = iforest.decision_function(X_test_scaled)
print("Test anomaly score (decision_function) min/max:", anom_test.min(), anom_test.max())

joblib.dump({"scaler": scaler, "iforest": iforest, "numeric_cols": numeric_cols}, MODEL_DIR / "isolation_forest.pkl")
print(f"Saved to {MODEL_DIR / 'isolation_forest.pkl'}")

Test anomaly score (decision_function) min/max: -0.11559278886866586 0.21585673979714742
Saved to /Users/zhumiban/Desktop/agent_bank/fraud_detection/models/isolation_forest.pkl


## Combined logic: CatBoost safe + IF anomalous → human review

In [4]:
# Use calibrated model from 03; otherwise scale_pos_weight inflates probs and 'safe' zone misses few
calibrated = joblib.load(MODEL_DIR / "catboost_calibrated.pkl")
cb_prob = calibrated.predict_proba(X_test)[:, 1]

cb_threshold = 0.5
if_review_percentile = 99  # top 1% anomaly score triggers review
if_threshold = np.percentile(anom_test, 100 - if_review_percentile)

cb_safe = cb_prob < cb_threshold
if_anomalous = anom_test <= if_threshold
review = cb_safe & if_anomalous

print(f"CatBoost threshold: {cb_threshold}")
print(f"IF review threshold (decision_function <=): {if_threshold:.4f}")
print(f"Test set: {review.sum()} samples flagged for human review (CB safe but IF anomalous)")
if review.sum() > 0:
    print(f"  Among these, actual fraud: {y_test[review].sum()}")

CatBoost threshold: 0.5
IF review threshold (decision_function <=): -0.0222
Test set: 1058 samples flagged for human review (CB safe but IF anomalous)
  Among these, actual fraud: 3
